# Prepare training examples for pretrained-embedding retrieval

This notebook prepares training data only. It does not fit a retrieval layer or fine-tune a transformer. It reuses existing frozen Gemini embeddings by default. BERT embeddings are also supported through explicit path overrides.

Sample up to 50,000 complete training requests using seed 42. For each usable clicked article, save its request's past-history profile reference and up to four displayed-but-unclicked article IDs. Every clicked article in that request is excluded from negatives. No validation/test behaviour labels are read. The shared train/test article lookup union is used only for fixed article metadata/vectors.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.util
import json
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
EXPERIMENT_ID = '3.3_pretrained_retrieval'
if IN_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    CODE_DIR = Path('/content/Experiments') / EXPERIMENT_ID
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    print('Select BOTH training_examples.py and catalogue_pipeline.py from this experiment folder.')
    uploaded = files.upload()
    for module in ['training_examples', 'catalogue_pipeline']:
        matches = [name for name in uploaded if Path(name).stem.startswith(module) and Path(name).suffix == '.py']
        if len(matches) != 1:
            raise ValueError(f'Select exactly one {module}.py file (Colab-renamed filenames are accepted).')
        (CODE_DIR / f'{module}.py').write_bytes(uploaded[matches[0]])
else:
    choices = [Path.cwd(), Path.cwd() / 'Experiments' / EXPERIMENT_ID]
    CODE_DIR = next((p for p in choices if (p / 'training_examples.py').is_file()), None)
    if CODE_DIR is None:
        raise FileNotFoundError('Open from the repository root or the experiment folder.')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy>=1.26', 'pandas>=2.2',
                'pyarrow>=15', 'scikit-learn>=1.4'], check=True)
import pandas as pd

def load_module(name):
    spec = importlib.util.spec_from_file_location(name, CODE_DIR / f'{name}.py')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
cp = load_module('catalogue_pipeline')
te = load_module('training_examples')
print('Loaded:', cp.__file__, te.__file__)


## Configuration

Start with the shared Gemini embeddings. To reuse generated BERT vectors, override only `embeddings_train_val` and `embeddings_test` with the actual saved parquet paths and record the encoder/text variant below. Do not regenerate embeddings. Keep this dataset run separate from previous similarity run folders. CPU is sufficient for preparation; a GPU will be useful for later model fitting.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')
RUN_ID = datetime.now(ZoneInfo('Asia/Singapore')).strftime('%Y%m%d_%H%M%S') + '_training_examples_seed42'
WORK = CODE_DIR / 'outputs' / RUN_ID
CONFIG = {
    'experiment_id': EXPERIMENT_ID,
    'run_id': RUN_ID,
    'drive_root': str(DRIVE_ROOT),
    'seed': 42,
    'train_requests': 50_000,
    'negatives_per_positive': 4,
    'history_limit': 10,
    'embedding_column': 'text_embedding',
    'embedding_dimension': None,
    'embedding_source': {'provider': 'Gemini', 'model': None, 'revision': None,
                         'note': 'Shared frozen embeddings; confirm encoder identity with preprocessing owner.'},
    'max_index_gb': 20,
    'hash_inputs': False,
    'paths': {},
}
# For BERT, use the EXACT embedding files from your existing embedding cache:
# CONFIG['paths']['embeddings_train_val'] = '/content/drive/MyDrive/.../train_val_text_embeddings.parquet'
# CONFIG['paths']['embeddings_test'] = '/content/drive/MyDrive/.../test_text_embeddings.parquet'
# CONFIG['embedding_source'] = {'provider': 'Sentence Transformers',
#     'model': 'sentence-transformers/distiluse-base-multilingual-cased-v2',
#     'revision': 'COPY_FROM_embedding_config.json', 'text_variant': 'title_subtitle'}
print('Drive root exists:', DRIVE_ROOT.exists())
print('Dataset output:', WORK)


## Check sources and index budget

Only the training feature table is indexed. This still scans the existing training pairs once, but does not create catalogue-wide negatives. The requested sample size counts complete requests, not pair rows or positive examples. One request can create several examples when it has multiple clicked articles. Missing profiles/vectors and publication-ineligible training candidates are excluded from examples and counted; this does not alter inference eligibility or evaluation denominators.


In [ ]:
SOURCES = te.resolve_sources(CONFIG)
for name, path in SOURCES.items():
    print(name, '→', path)
train_info = cp.fingerprint(SOURCES['features_train'])
print('Recorded training rows:', train_info['rows'])
print('Conservative generated-index estimate (GB):', round(train_info['rows'] * 320 / 1e9, 2))


## Build the dataset

The loader uses the same deterministic complete-request sampling rule as the similarity experiments. Profiles use a normalized mean of the latest 10 available past-click article vectors. The fixed source history must predate the earliest request in the entire training split, not just the sampled requests. Training labels supply positives/negatives only; they are never inserted into history profiles.

Negative sampling is reproducible for each request/positive. If fewer than four usable displayed negatives exist, keep fewer. Requests with no usable profile or no usable negative cannot supply this training objective. No random unshown catalogue article is assigned a negative label.


In [ ]:
SUMMARY = te.build(CONFIG, WORK, cp)
display(pd.json_normalize(SUMMARY['counts']))
print('Saved users:', SUMMARY['saved_users'])
print('Saved articles:', SUMMARY['saved_articles'])
print('Model training performed:', SUMMARY['model_training_performed'])


## Inspect output

`training_examples.parquet` stores request IDs, one positive ID and a list of up to four negative IDs. Join `user_id` to `user_profiles.parquet`; join article IDs to `article_vectors.parquet`. Numeric vectors are stored once rather than repeated for every example. Fewer-negative examples need a padding mask in the later ranking loss. Displayed-but-unclicked articles are imperfect negative signals. Multiple positives from the same request are NEVER sampled against each other.


In [ ]:
import pyarrow.parquet as pq
first_batch = next(pq.ParquetFile(WORK / 'training_examples.parquet').iter_batches(batch_size=5))
display(first_batch.to_pandas())
print('Files:')
for name in te.ARTIFACTS:
    path = WORK / name
    print(name, round(path.stat().st_size / 1e6, 2), 'MB')


## Save to Drive

Copy the portable dataset and its configuration/source manifests; omit generated SQLite indexes. Existing destinations are not overwritten. If a copy was interrupted, choose a fresh destination suffix rather than deleting previous results. Keep WORK/RUN_ID unchanged when rerunning inspection or copying.


In [ ]:
DESTINATION = DRIVE_ROOT / 'Experiments' / EXPERIMENT_ID / RUN_ID
SAVED = te.persist(WORK, DESTINATION)
print('Saved training dataset:', SAVED)


## Next step

The output is ready to train a small user/article projection model over frozen vectors. That later model should learn to score the positive above its sampled displayed alternatives, using a masked ranking loss. Use validation catalogue Recall@200 to select checkpoints. Keep test labels out of model selection. This notebook deliberately reports dataset counts rather than model accuracy because no weights have been fitted yet.
